# Colombia Open — Geometría de municipios

Este notebook muestra cómo cargar el dataset **`municipios_geometria`** (geometría de polígonos municipales, Marco Geoestadístico Nacional del DANE vía HDX) con `colombia-open` — segunda iteración de `divipola`/`municipios`, que solo trae centroide.

> **Nota:** `geometry_wkt` es texto plano (WKT) — este notebook usa `shapely` para convertirlo a geometría y `geopandas` para graficar, ambos ya disponibles en Colab. Leer el dataset publicado NO requiere el extra `geo` de `colombia-open` (ese extra es solo para correr el extractor localmente).

> **Nota (2026-10):** `colombia-open` está publicado en [PyPI](https://pypi.org/project/colombia-open/) — `pip install colombia-open` descarga y cachea el bundle de datos publicado en GitHub Releases automáticamente. Este dataset está disponible desde el bundle `v0.8.0`.

## Setup

In [ ]:
!pip install -q colombia-open shapely geopandas matplotlib

## Cargar el dataset

In [ ]:
import polars as pl
from colombia_open import ColombiaOpen

hub = ColombiaOpen()
geometria = hub.load_polars("municipios_geometria")

print(geometria.shape)
geometria.drop("geometry_wkt").head(10)

## Esquema

| Columna | Tipo | Descripción |
|---|---|---|
| `cod_dpto` / `cod_mpio` | string (2 / 5 dígitos) | Códigos DIVIPOLA — join key hacia `municipios`. |
| `geom_type` | string | `"Polygon"` (1113 municipios) o `"MultiPolygon"` (9, islas/exclaves). |
| `area_km2` | float | Área geodésica, calculada sobre la geometría original antes de simplificar. |
| `geometry_wkt` | string | Polígono en WKT, EPSG:4326, **simplificado** (~55m de tolerancia) — no es survey-grade. |

Clave primaria: `cod_mpio`. Esquema completo: [`docs/datasets/municipios_geometria.md`](../../docs/datasets/municipios_geometria.md).

## Los 10 municipios más grandes por área

In [ ]:
geometria.sort("area_km2", descending=True).select(["cod_mpio", "geom_type", "area_km2"]).head(10)

## Graficar un municipio (join con `municipios` para el nombre)

In [ ]:
import geopandas as gpd
import shapely.wkt

municipios = hub.load_polars("municipios")
amazonas = geometria.join(municipios.select(["cod_mpio", "nombre_mpio"]), on="cod_mpio").filter(
    pl.col("cod_mpio") == "91001"  # Leticia, el municipio más grande del país
)

gdf = gpd.GeoDataFrame(
    amazonas.to_pandas(),
    geometry=amazonas["geometry_wkt"].map_elements(shapely.wkt.loads, return_dtype=pl.Object).to_pandas(),
    crs="EPSG:4326",
)
gdf.plot(figsize=(6, 6), edgecolor="black")

## Densidad poblacional (join con `demografia`)

Población total 2024 dividida por el área del municipio.

In [ ]:
poblacion = hub.load_polars("demografia").filter(
    (pl.col("area_geografica") == "Total") & (pl.col("anio") == 2024)
)

(
    geometria.join(poblacion.select(["cod_mpio", "poblacion"]), on="cod_mpio")
    .join(municipios.select(["cod_mpio", "nombre_mpio"]), on="cod_mpio")
    .with_columns((pl.col("poblacion") / pl.col("area_km2")).round(1).alias("densidad_hab_km2"))
    .sort("densidad_hab_km2", descending=True)
    .select(["nombre_mpio", "poblacion", "area_km2", "densidad_hab_km2"])
    .head(10)
)

## Limitaciones conocidas

- **Geometría simplificada, no survey-grade** — tolerancia Douglas-Peucker de ~55m, pensada para visualización, no para análisis catastral.
- **No incluye nombre de municipio** — hace falta el join con `municipios` (ver celdas arriba).
- **Vigencia 2020-04-16** del Marco Geoestadístico Nacional — 1 código de municipio en cada sentido no coincide exactamente con el `municipios` vigente (documentado, warning no error).

Documentación completa: [`docs/datasets/municipios_geometria.md`](../../docs/datasets/municipios_geometria.md).